# Task 4 — Neural Feynman–Kac regression
Train a neural network from freshly simulated stochastic terminal payoffs. Squared-loss regression targets the conditional expectation, i.e. the Black–Scholes solution curve.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from src.neural_solver import train_neural_feynman_kac, predict_prices, error_metrics
from src.monte_carlo import black_scholes_call


In [ ]:
tr = train_neural_feynman_kac(dim=1,payoff='call',low=50,high=150,strike=100,r=0.05,sigma=0.2,T=1.0,steps=1800,batch_size=2048,hidden=(96,96,96),seed=321,antithetic=True)
print(tr.history.tail())
print('training seconds:', tr.train_seconds)


In [ ]:
grid=np.linspace(50,150,401)[:,None]
pred=predict_prices(tr.model,grid,100)
truth=black_scholes_call(grid[:,0],100,0.05,0.2,1.0)
print(error_metrics(pred,truth))
plt.figure(figsize=(6,4))
plt.plot(grid[:,0],truth,label='Black–Scholes')
plt.plot(grid[:,0],pred,'--',label='neural FK')
plt.xlabel('S'); plt.ylabel('V(0,S)'); plt.legend(); plt.grid(True,alpha=.25);


In [ ]:
rng=np.random.default_rng(77)
x=rng.uniform(50,150,800); z=rng.standard_normal(800)
st=x*np.exp((0.05-.5*0.2**2)+0.2*z)
y=np.exp(-0.05)*np.maximum(st-100,0)
plt.figure(figsize=(6,4))
plt.scatter(x,y,s=7,alpha=.2,label='single-path labels')
plt.plot(grid[:,0],truth,label='conditional mean')
plt.plot(grid[:,0],pred,'--',label='neural FK')
plt.ylim(-2,110); plt.legend(); plt.grid(True,alpha=.2);
